In [1]:
import sys
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import json
import random
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import yaml
import cv2
from datasets.fmd.data_loader import load_noisy_single_loader
from evaluation import get_largest_residual_indices
from evaluation.calibration_utils import (AnalysisManager, CalibrationManager,
                                          get_test_dataloader)
from models.checkpointing import load_checkpoint_for_inference, create_model, load_model_state
from QUTCC_eval import plot_visualization, create_single_pixel_pdf_plot, plot_vis_slider
from train import get_dataloaders, get_transform
from omegaconf import OmegaConf

np.random.seed(0)
random.seed(0)
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
# print("tmp folder: ", os.environ["TMPDIR"])

%matplotlib inline
%load_ext autoreload
%autoreload 2

/home/bl788/.conda/envs/qutcc2/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/bl788/.conda/envs/qutcc2/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an 

Device: cuda


In [2]:
#Load in weights + calibration values
im2im_deep_folder = "experiments/mri/im2im_deep_epochs50_bs12_2025-10-18-14.18.46"

qutcc_folder = "experiments/mri/qutcc_epochs50_bs12_2025-10-18-10.07.36"

im2im_ckpt = os.path.join(im2im_deep_folder, 'checkpoints', 'epoch10.pth')
qutcc_ckpt = os.path.join(qutcc_folder, 'checkpoints', 'epoch10.pth')

# load jsons from the directories
with open(os.path.join(qutcc_folder, 'analysis', 'quantile_qs_epoch_5.json'), 'r') as f:
    qutcc_config = json.load(f)

# extracted this manually as the error bound was too high despite running for 10 epochs and failed to calculate bound
im2im_deep_lambda = 1.0110
upper_q = float(qutcc_config['upper_q'])
lower_q = float(qutcc_config['lower_q'])

im2im_deep = create_model(net = "im2im_deep", device = device)
im2im_deep = load_model_state(im2im_deep, im2im_ckpt, device)

qutcc = create_model(net = "qutcc", device = device)
qutcc = load_model_state(qutcc, qutcc_ckpt, device)

Model state loaded from experiments/mri/im2im_deep_epochs50_bs12_2025-10-18-14.18.46/checkpoints/epoch10.pth
Model state loaded from experiments/mri/qutcc_epochs50_bs12_2025-10-18-10.07.36/checkpoints/epoch10.pth


In [3]:
args = OmegaConf.create({
    "net": "im2im_deep",
    "transform": "center_crop",
    "epochs": 50,
    "experiment_type": "MRI",
    "data_root": "/share/monakhova/Cassandra_data/UQNet_proj/Fast_MRI/RAW_singlecoil_train",
    "in_channels": 1,
    "noise_type": "poisson",
    "sigma": 0.75,
    "batch_size": 12,
    "exp_name": "mri",
    "ckpt_freq": 2
})

_, test_loader = get_dataloaders(args, get_transform(args))

loading dataset from /share/monakhova/Cassandra_data/UQNet_proj/Fast_MRI/RAW_singlecoil_train/training...
Loading 644 volumes...
Using 22838 total slices
Experiment type: MRI | Train dataset size: 20554 | Test dataset size: 2284


In [5]:
#Visualization
plot_vis_slider(dataloader=test_loader, im2im_model=im2im_deep, 
                    quantile_model=qutcc, im2im_lam=im2im_deep_lambda, lower_q=lower_q, upper_q=upper_q, 
                    device=device, exp_type="exp", save=False)

Loading images from dataloader...
Loaded 2284 images from dataloader


interactive(children=(IntSlider(value=0, continuous_update=False, description='Image:', max=2283), Output()), …

<function QUTCC_eval.plot_vis_slider.<locals>.plot_for_index(index)>